# eBPF Overhead Measurement

Produces TABLE_3 from the paper: CPU/memory overhead, event loss rate, collection latency.

In [ ]:
# Cell 1: Load Prometheus metrics from experiment runs
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import glob

DATASET_DIR = Path('research/datasets/phantom-v1')
RESULTS_DIR = Path('research/datasets/raw')

traces = pd.read_parquet(DATASET_DIR / 'traces.parquet')

# Load all scenario results for phase-level metrics
scenario_results = []
for f in sorted(glob.glob(str(RESULTS_DIR / '*.json'))):
    if 'index' in f:
        continue
    try:
        with open(f) as fh:
            r = json.load(fh)
        scenario_results.append(r)
    except Exception:
        pass

print(f"Loaded {len(scenario_results)} scenario results.")

# Extract metrics snapshots
metric_rows = []
for r in scenario_results:
    for snap in r.get('metrics_snapshots', []):
        metric_rows.append({
            'run_id': r['run_id'],
            'attack_family': r['attack_family'],
            'ground_truth_label': r['ground_truth_label'],
            'phase': snap['phase'],
            'cpu_usage_cores': snap['cpu_usage_cores'],
            'memory_rss_mb': snap['memory_rss_mb'],
            'event_lag_p95_ms': snap['event_lag_p95_ms'],
        })

metrics_df = pd.DataFrame(metric_rows)
print(metrics_df.groupby('phase')[['cpu_usage_cores', 'memory_rss_mb', 'event_lag_p95_ms']].describe().round(3))


In [ ]:
# Cell 2: CPU overhead table — TABLE_3

# Compare baseline vs attack phase CPU
baseline_cpu = metrics_df[metrics_df['phase'] == 'baseline']['cpu_usage_cores']
attack_cpu   = metrics_df[metrics_df['phase'] == 'attack']['cpu_usage_cores']

cpu_overhead_pct = 100 * (attack_cpu.mean() - baseline_cpu.mean()) / max(baseline_cpu.mean(), 1e-9)

print("=== TABLE_3: Overhead Summary ===")
print(f"Baseline CPU (cores):   {baseline_cpu.mean():.4f} ± {baseline_cpu.std():.4f}")
print(f"Attack phase CPU:       {attack_cpu.mean():.4f} ± {attack_cpu.std():.4f}")
print(f"CPU overhead:           {cpu_overhead_pct:.2f}%")

baseline_mem = metrics_df[metrics_df['phase'] == 'baseline']['memory_rss_mb']
attack_mem   = metrics_df[metrics_df['phase'] == 'attack']['memory_rss_mb']
print(f"Baseline Memory (MB):   {baseline_mem.mean():.2f} ± {baseline_mem.std():.2f}")
print(f"Attack phase Memory:    {attack_mem.mean():.2f} ± {attack_mem.std():.2f}")
print(f"Memory overhead (MB):   {(attack_mem.mean() - baseline_mem.mean()):.2f}")

lag_col = metrics_df[metrics_df['phase'] == 'attack']['event_lag_p95_ms']
print(f"Event lag P95 (ms):     {lag_col.mean():.2f} ± {lag_col.std():.2f}")


In [ ]:
# Cell 3: Memory overhead bar chart

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CPU by phase
phase_order = ['baseline', 'attack', 'post_recovery']
cpu_by_phase = metrics_df.groupby('phase')['cpu_usage_cores'].mean().reindex(
    [p for p in phase_order if p in metrics_df['phase'].unique()]
)
axes[0].bar(cpu_by_phase.index, cpu_by_phase.values, color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0].set_xlabel('Scenario Phase')
axes[0].set_ylabel('CPU Usage (cores)')
axes[0].set_title('PHANTOM Agent CPU by Phase')

# Memory by phase
mem_by_phase = metrics_df.groupby('phase')['memory_rss_mb'].mean().reindex(
    [p for p in phase_order if p in metrics_df['phase'].unique()]
)
axes[1].bar(mem_by_phase.index, mem_by_phase.values, color=['#3498db', '#e74c3c', '#2ecc71'])
axes[1].set_xlabel('Scenario Phase')
axes[1].set_ylabel('Memory RSS (MB)')
axes[1].set_title('PHANTOM Agent Memory by Phase')

plt.tight_layout()
plt.savefig('research/evaluation/results/table_3_overhead.pdf', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 4: Event lag distribution histogram

lag_values = metrics_df['event_lag_p95_ms'].dropna()

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(lag_values, bins=30, color='#2ecc71', edgecolor='black', alpha=0.8)
ax.axvline(lag_values.quantile(0.95), color='red', linestyle='--',
           label=f'P95 = {lag_values.quantile(0.95):.1f} ms')
ax.axvline(lag_values.median(), color='blue', linestyle='-.',
           label=f'Median = {lag_values.median():.1f} ms')
ax.set_xlabel('Event Collection Latency P95 (ms)')
ax.set_ylabel('Count')
ax.set_title('eBPF Event Collection Latency Distribution\n(t_ingest - t_kernel)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('research/evaluation/results/overhead_event_lag.pdf', dpi=150, bbox_inches='tight')
plt.show()
print(f"P50 lag: {lag_values.quantile(0.50):.2f} ms")
print(f"P95 lag: {lag_values.quantile(0.95):.2f} ms")
print(f"P99 lag: {lag_values.quantile(0.99):.2f} ms")


In [ ]:
# Cell 5: Ring buffer loss rate

# ringbuf_lost_delta from traces.parquet
if 'ringbuf_lost_delta' in traces.columns and traces['ringbuf_lost_delta'].notna().any():
    total_events = len(traces)
    total_lost = traces['ringbuf_lost_delta'].dropna().sum()
    loss_rate = total_lost / max(total_events + total_lost, 1)
    print(f"Ring buffer loss rate: {loss_rate:.4%}")
    print(f"Total events processed: {total_events:,}")
    print(f"Total events lost:      {total_lost:,.0f}")
else:
    print("No ringbuf_lost_delta data in traces (column is null).")
    print("Verify eBPF agent exposes phantom_ebpf_ringbuf_reserve_failures_total.")
